In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error
import lightgbm as lgb

In [ ]:
train = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/train.csv", parse_dates=['date'])
test = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/test.csv", parse_dates=['date'])
stores = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/stores.csv")
transactions = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/transactions.csv", parse_dates=['date'])
holidays = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv", parse_dates=['date'])
oil = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/oil.csv", parse_dates=['date'])

In [ ]:
holidays_renamed = holidays[["date", "type"]].rename(columns={"type": "holiday_type"})

In [ ]:
train = train.merge(stores, on="store_nbr", how="left")
test = test.merge(stores, on="store_nbr", how="left")

In [ ]:
train = train.merge(oil, on="date", how="left")
test = test.merge(oil, on="date", how="left")

In [ ]:
train = train.merge(transactions, on=["store_nbr", "date"], how="left")
test = test.merge(transactions, on=["store_nbr", "date"], how="left")

In [ ]:
train = train.merge(holidays_renamed, on="date", how="left")
test = test.merge(holidays_renamed, on="date", how="left")

In [ ]:
train["holiday_type"] = train["holiday_type"].fillna("None")
test["holiday_type"] = test["holiday_type"].fillna("None")

In [ ]:
for df in [train, test]:
    df["day"] = df["date"].dt.day
    df["month"] = df["date"].dt.month
    df["year"] = df["date"].dt.year
    df["day_of_week"] = df["date"].dt.dayofweek
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

cat_cols = train.select_dtypes(include=["object"]).columns.tolist()
for col in cat_cols:
    for df in [train, test]:
        df[col] = df[col].astype("category").cat.codes

In [ ]:
target = "sales"
features = [col for col in train.columns if col not in ["id", "date", "sales"]]

X = train[features]
y = train[target]
X_test = test[features]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, shuffle=False)


In [ ]:
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val)

params = {
    "objective": "regression",
    "metric": "rmse",  
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "seed": 42
}



In [ ]:
from lightgbm import LGBMRegressor, early_stopping

model = LGBMRegressor(
    objective="regression",
    learning_rate=0.05,
    num_leaves=31,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    random_state=42,
    n_estimators=1000
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="rmse",
    callbacks=[early_stopping(stopping_rounds=50)]
)


In [ ]:
val_preds = model.predict(X_val)
val_preds = np.maximum(0, val_preds)

from sklearn.metrics import mean_squared_log_error
rmsle_score = np.sqrt(mean_squared_log_error(y_val, val_preds))
print("Validation RMSLE:", rmsle_score)

test_preds = model.predict(X_test)
test_preds = np.maximum(0, test_preds)

submission = pd.DataFrame({
    "id": test["id"],
    "sales": test_preds
})
submission.to_csv("lgbm_submission.csv", index=False)